# Phase 1: Acquire, Reshape and Clean
## World Bank Financial Sector Indicators

### What This Notebook Does
- Downloads 8 financial indicators for ~200 countries (2000-2022)
- Downloads country metadata (region and income group)
- Inspects the raw data
- Saves to data/raw/ folder

In [1]:
import wbgapi as wb
import pandas as pd

print(f'wbgapi version: {wb.__version__}')
print(f'pandas version: {pd.__version__}')
print('Libraries loaded successfully')

wbgapi version: 1.0.14
pandas version: 2.2.2
Libraries loaded successfully


In [8]:
# Search 1 - find account ownership indicator
wb.series.info(q='account ownership')

id,value
FX.OWN.TOTL.40.ZS,"Account ownership at a financial institution or with a mobile-money-service provider, poorest 40% (% of population ages 15+)"
FX.OWN.TOTL.60.ZS,"Account ownership at a financial institution or with a mobile-money-service provider, richest 60% (% of population ages 15+)"
FX.OWN.TOTL.FE.ZS,"Account ownership at a financial institution or with a mobile-money-service provider, female (% of population ages 15+)"
FX.OWN.TOTL.MA.ZS,"Account ownership at a financial institution or with a mobile-money-service provider, male (% of population ages 15+)"
FX.OWN.TOTL.OL.ZS,"Account ownership at a financial institution or with a mobile-money-service provider, older adults (% of population ages 25+)"
FX.OWN.TOTL.PL.ZS,"Account ownership at a financial institution or with a mobile-money-service provider, primary education or less (% of population ages 15+)"
FX.OWN.TOTL.SO.ZS,"Account ownership at a financial institution or with a mobile-money-service provider, secondary education or more (% of population ages 15+)"
FX.OWN.TOTL.YG.ZS,"Account ownership at a financial institution or with a mobile-money-service provider, young adults (% of population ages 15-24)"
FX.OWN.TOTL.ZS,Account ownership at a financial institution or with a mobile-money-service provider (% of population ages 15+)
,9 elements


chose FX.OWN.TOTL.ZS -> account ownership

In [9]:
# Search 2 - find nonperforming loans indicator
wb.series.info(q='nonperforming loans')

id,value
FB.AST.NPER.ZS,Bank nonperforming loans to total gross loans (%)
,1 elements


In [11]:
# Search 3 - find bank branches indicator
wb.series.info(q='bank branches')

id,value
FB.CBK.BRCH.P5,"Commercial bank branches (per 100,000 adults)"
,1 elements


In [19]:
# Search 4
wb.series.info(q='domestic credit')

id,value
FD.AST.PRVT.GD.ZS,Domestic credit to private sector by banks (% of GDP)
FM.AST.DOMS.CN,Net domestic credit (current LCU)
FS.AST.DOMS.GD.ZS,Domestic credit provided by financial sector (% of GDP)
FS.AST.PRVT.GD.ZS,Domestic credit to private sector (% of GDP)
,4 elements


Chose FS.AST.PRVT.GD.ZS	#best for comparing financial depth across countries

In [14]:
# Search 5
wb.series.info(q='lending interest rate')

id,value
FR.INR.LEND,Lending interest rate (%)
,1 elements


In [20]:
# Search 6
wb.series.info(q='deposit interest rate')

id,value
FR.INR.DPST,Deposit interest rate (%)
,1 elements


In [21]:
# Search 7
wb.series.info(q='bank capital')

id,value
FB.BNK.CAPA.ZS,Bank capital to assets ratio (%)
,1 elements


In [17]:
# Search 8
wb.series.info(q='GDP per capita')

id,value
NY.GDP.PCAP.CD,GDP per capita (current US$)
NY.GDP.PCAP.CN,GDP per capita (current LCU)
NY.GDP.PCAP.KD,GDP per capita (constant 2015 US$)
NY.GDP.PCAP.KD.ZG,GDP per capita growth (annual %)
NY.GDP.PCAP.KN,GDP per capita (constant LCU)
NY.GDP.PCAP.PP.CD,"GDP per capita, PPP (current international $)"
NY.GDP.PCAP.PP.KD,"GDP per capita, PPP (constant 2021 international $)"
,7 elements


Chose NY.GDP.PCAP.CD

In [24]:
# Final 8 indicators chosen after searching the World Bank database
indicators = {
    'FS.AST.PRVT.GD.ZS': 'Credit to Private Sector pct GDP',
    'FB.AST.NPER.ZS':     'Nonperforming Loans pct',
    'FR.INR.LEND':        'Lending Interest Rate pct',
    'FR.INR.DPST':        'Deposit Interest Rate pct',
    'FB.CBK.BRCH.P5':     'Bank Branches per 100k Adults',
    'FX.OWN.TOTL.ZS':     'Account Ownership pct Adults',
    'FB.BNK.CAPA.ZS':     'Bank Capital to Assets pct',
    'NY.GDP.PCAP.CD':     'GDP per Capita USD'
}

print('Final indicators selected:')
for code, name in indicators.items():
    print(f'  {code}  ->  {name}')

print(f'\nTotal indicators: {len(indicators)}')

Final indicators selected:
  FS.AST.PRVT.GD.ZS  ->  Credit to Private Sector pct GDP
  FB.AST.NPER.ZS  ->  Nonperforming Loans pct
  FR.INR.LEND  ->  Lending Interest Rate pct
  FR.INR.DPST  ->  Deposit Interest Rate pct
  FB.CBK.BRCH.P5  ->  Bank Branches per 100k Adults
  FX.OWN.TOTL.ZS  ->  Account Ownership pct Adults
  FB.BNK.CAPA.ZS  ->  Bank Capital to Assets pct
  NY.GDP.PCAP.CD  ->  GDP per Capita USD

Total indicators: 8


In [25]:
# Check what years are available for one indicator
wb.series.info('FX.OWN.TOTL.ZS')

id,value
FX.OWN.TOTL.ZS,Account ownership at a financial institution or with a mobile-money-service provider (% of population ages 15+)
,1 elements


In [27]:
# Check latest available year for account ownership
wb.data.DataFrame('FX.OWN.TOTL.ZS', time=range(2020, 2027)).tail()

,YR2020,YR2021,YR2022,YR2023,YR2024,YR2025
economy,,,,,,
XKX,NaN,58.011960,NaN,NaN,64.161772,NaN
YEM,NaN,NaN,11.903366,NaN,NaN,NaN
ZAF,NaN,85.378112,NaN,NaN,81.125966,NaN
ZMB,NaN,48.521887,NaN,NaN,72.702425,NaN
ZWE,NaN,59.745929,NaN,NaN,49.521602,NaN


In [28]:
# Download all 8 indicators for all countries, years 2000 to 2022
print('Downloading data from World Bank...')

raw_df = wb.data.DataFrame(
    list(indicators.keys()),
    time=range(2000, 2025),
    labels=True
)

print(f'\nDownload complete')
print(f'Shape: {raw_df.shape}')
print(f'\nColumn names:')
print(raw_df.columns.tolist())
print(f'\nFirst 5 rows:')
raw_df.head()


Download complete
Shape: (2120, 27)

Column names:
['Country', 'Series', 'YR2000', 'YR2001', 'YR2002', 'YR2003', 'YR2004', 'YR2005', 'YR2006', 'YR2007', 'YR2008', 'YR2009', 'YR2010', 'YR2011', 'YR2012', 'YR2013', 'YR2014', 'YR2015', 'YR2016', 'YR2017', 'YR2018', 'YR2019', 'YR2020', 'YR2021', 'YR2022', 'YR2023', 'YR2024']

First 5 rows:


,,Country,Series,YR2000,YR2001,YR2002,YR2003,YR2004,YR2005,YR2006,YR2007,...,YR2015,YR2016,YR2017,YR2018,YR2019,YR2020,YR2021,YR2022,YR2023,YR2024
economy,series,,,,,,,,,,,,,,,,,,,,,
ZWE,FS.AST.PRVT.GD.ZS,Zimbabwe,Domestic credit to private sector (% of GDP),21.562896,27.724546,84.813370,53.168307,17.047969,14.948549,NaN,NaN,...,18.315744,17.098496,5.810054,5.830267,3.428022,3.642132,4.759522,5.826323,6.477697,NaN
ZMB,FS.AST.PRVT.GD.ZS,Zambia,Domestic credit to private sector (% of GDP),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,19.764834,15.442121,14.906011,14.973719,15.626190,15.163555,11.359504,12.975089,15.626306,17.007380
YEM,FS.AST.PRVT.GD.ZS,"Yemen, Rep.",Domestic credit to private sector (% of GDP),4.860630,5.734766,5.801288,6.366398,7.162536,6.939312,6.913220,8.184085,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PSE,FS.AST.PRVT.GD.ZS,West Bank and Gaza,Domestic credit to private sector (% of GDP),21.488655,21.410608,24.196521,21.693138,23.117607,26.446226,26.217355,22.867428,...,32.601047,36.527932,41.742378,44.840288,44.926735,52.243821,47.329516,47.710617,52.732572,58.625272
VIR,FS.AST.PRVT.GD.ZS,Virgin Islands (U.S.),Domestic credit to private sector (% of GDP),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
# Download country metadata
# This gives us region and income group for every country
print('Downloading country metadata...')

metadata = wb.economy.DataFrame()

print(f'Metadata shape: {metadata.shape}')
print(f'\nColumns: {metadata.columns.tolist()}')
print(f'\nFirst 5 rows:')
metadata.head()

Metadata shape: (265, 9)

Columns: ['name', 'aggregate', 'longitude', 'latitude', 'region', 'adminregion', 'lendingType', 'incomeLevel', 'capitalCity']

First 5 rows:


,name,aggregate,longitude,latitude,region,adminregion,lendingType,incomeLevel,capitalCity
id,,,,,,,,,
ABW,Aruba,False,-70.0167,12.51670,LCN,,LNX,HIC,Oranjestad
AFE,Africa Eastern and Southern,True,NaN,NaN,,,,,
AFG,Afghanistan,False,69.1761,34.52280,MEA,MNA,IDX,LIC,Kabul
AFW,Africa Western and Central,True,NaN,NaN,,,,,
AGO,Angola,False,13.2420,-8.81155,SSF,SSA,IBD,LMC,Luanda


In [31]:
# Save raw downloaded data to raw folder
raw_df.to_csv('../data/raw/financial_indicators_raw.csv')
metadata.to_csv('../data/raw/country_metadata_raw.csv')

print(f'financial_indicators_raw.csv  ->  {raw_df.shape}')
print(f'country_metadata_raw.csv      ->  {metadata.shape}')

financial_indicators_raw.csv  ->  (2120, 27)
country_metadata_raw.csv      ->  (265, 9)


## Phase 1 Download Complete

### Files Saved to data/raw/
- financial_indicators_raw.csv: 2120 rows x 27 columns
- country_metadata_raw.csv: 265 rows x 9 columns

### Key Observations
- Data covers 2000 to 2024 (25 years)
- Wide format: countries as rows, years as columns
- Metadata contains 48 regional aggregates that must be removed
- Several NaN values visible especially in earlier years
- Region and income level codes need decoding in cleaning phase